# Tabular Q-Learning

## Knowledge Limitations for Optimal Policies
In the prior lecture, we evaluated policies and generated improvements by calculating exact Q-values. Calculating an exact Q-value for every state-action pair assumes complete knowledge of the transition probabilities, denoted as $P(s'|s,a)$, and the reward function, denoted as $R(s)$. In practical scenarios where we must approximate the Q-function, we typically do not know the underlying transition probabilities dictating how actions affect the state space. Additionally, exact calculations become unfeasible when state and action spaces are not small and discrete.

## The Bellman Equations
To formalize the recursive nature of these values, we define the Bellman equation for a given policy $\pi$. Using strictly Q-notation, the Bellman equation dictates that the expected value of taking an action $a$ in state $s$ is the immediate reward plus the discounted expected value of following policy $\pi$ thereafter:

$$Q_\pi(s,a) = R(s) + \gamma \sum_{s'} P(s'|s,a) Q_\pi(s', \pi(s'))$$

The optimal policy, which produces equal or greater discounted rewards compared to any alternative policy, satisfies the Bellman optimality equation. In this case, the future action is selected by maximizing the Q-value over all possible next actions:

$$Q^*(s,a) = R(s) + \gamma \sum_{s'} P(s'|s,a) \max_{a'} Q^*(s', a')$$

## Q-Function Approximation and Bellman Error
Because we typically lack access to the exact transition probabilities $P(s'|s,a)$ and the reward function $R(s)$ defining the environment, we cannot analytically compute the expected values over all possible future states. Instead, we must introduce an approximation of the Q-function, denoted as $\hat{Q}(s,a)$.

If $\hat{Q}(s,a)$ perfectly captures the optimal policy, it will perfectly satisfy the Bellman optimality equation. When it does not, the discrepancy between our current estimate and the expected target dictates the inaccuracy of our approximation. This difference is defined as the Bellman Error:

$$Error = \left( R(s) + \gamma \max_{a'} \hat{Q}(s', a') \right) - \hat{Q}(s,a)$$

## Minimizing Error via Tabular Q-Learning
Tabular Q-learning is a method designed to minimize this Bellman Error without requiring prior knowledge of the transition probabilities. Instead of computing expected values across all possible future states, the agent interacts with the environment and samples empirical transitions consisting of a state, an action, a resulting reward, and a new state $(s, a, r, s')$.

In tabular Q-learning, the Q-function approximation $\hat{Q}(s,a)$ is represented as a table with a separate entry for every possible state-action pair. This tabular representation is only feasible for MDPs with small, discrete state and action spaces, as the table size grows exponentially with the number of states and actions.

Upon observing a transition, the algorithm treats the observed reward plus the discounted maximum estimated future value as a target. The approximation $\hat{Q}(s,a)$ is then updated to incrementally reduce the Bellman Error using a learning rate parameter $\alpha$:

$$\hat{Q}(s,a) \leftarrow \hat{Q}(s,a) + \alpha \left( R(s) + \gamma \max_{a'} \hat{Q}(s', a') - \hat{Q}(s,a) \right)$$

This process iterates to continually minimize the Bellman Error and converge upon the optimal Q-values.

## Tabular Q-Learning Demonstration
To demonstrate tabular Q-learning, we construct a discrete Markov Decision Process (MDP) with $N=5$ states, indexed 0 through 4. The action space consists of two actions: moving left (0) or moving right (1). The reward function is deterministic, yielding a reward of 1 when entering state $s_1$, and 0 otherwise.

### Defining the Environment and Initialization
In tabular Q-learning, the Q-function approximation $\hat{Q}(s,a)$ is represented as a table with a separate entry for every possible state-action pair. We begin by initializing this table with random values.

In [3]:
import numpy as np

# Environment definitions
num_states = 5
num_actions = 2 # 0: Left, 1: Right

def step(state, action):
    """Simulates the environment transition."""
    if action == 0: # Left
        next_state = max(0, state - 1)
    else: # Right
        next_state = min(num_states - 1, state + 1)
    
    # Reward of 1 for reaching state 1
    reward = 1.0 if state == 1 else 0.0
    return next_state, reward

# Initialize a random Q-table
np.random.seed(42)
Q = np.random.rand(num_states, num_actions)

print("Initial Random Q-Table:")
print(Q)

Initial Random Q-Table:
[[0.37454012 0.95071431]
 [0.73199394 0.59865848]
 [0.15601864 0.15599452]
 [0.05808361 0.86617615]
 [0.60111501 0.70807258]]


### A Single Transition and Update Step
The agent interacts with the environment and samples empirical transitions consisting of a state, an action, a resulting reward, and a new state $(s, a, r, s')$. Here, we specify a single transition explicitly and walk through the application of the update rule.The approximation $\hat{Q}(s,a)$ is updated to incrementally reduce the Bellman Error using a learning rate parameter $\alpha$.

In [5]:
# Hyperparameters
alpha = 0.1  # Learning rate
gamma = 0.9  # Discount factor

# Explicitly defining a single transition
s = 1
a = 0 # Moving left
s_prime, r = step(s, a)

print(f"Sampled Transition: s={s}, a={a}, r={r}, s'={s_prime}")

# Calculate the target and apply the update
q_current = Q[s, a]
q_target = r + gamma * np.max(Q[s_prime])

# Applying the update equation
# Q(s,a) <- Q(s,a) + alpha * (r + gamma * max(Q(s', a')) - Q(s,a))
q_new = q_current + alpha * (q_target - q_current)

print(f"Pre-update Q({s}, {a}): {q_current:.4f}")
Q[s, a] = q_new
print(f"Post-update Q({s}, {a}): {Q[s, a]:.4f}")

Sampled Transition: s=1, a=0, r=1.0, s'=0
Pre-update Q(1, 0): 0.7320
Post-update Q(1, 0): 0.8444


### Iterating Until Convergence
This process iterates to continually minimize the Bellman Error and converge upon the optimal Q-values. We simulate this by continuously sampling random transitions and applying the update rule over many iterations.

In [6]:
# Iterating to convergence
epochs = 10000

for _ in range(epochs):
    # Sample a random state and action to explore the environment
    curr_s = np.random.randint(0, num_states)
    curr_a = np.random.randint(0, num_actions)
    
    # Observe the result of the action
    next_s, curr_r = step(curr_s, curr_a)
    
    # Update the Q-table using the Q-learning algorithm
    current_q = Q[curr_s, curr_a]
    max_future_q = np.max(Q[next_s])
    
    Q[curr_s, curr_a] = current_q + alpha * (curr_r + gamma * max_future_q - current_q)

print("Converged Q-Table:")
print(np.round(Q, 4))

Converged Q-Table:
[[4.263  4.7367]
 [5.263  5.2631]
 [4.7367 3.8367]
 [4.2631 3.4531]
 [3.8367 3.4531]]


## Deploying the Optimal Policy
Once the Q-table converges to the optimal values, executing optimal behavior requires very little deploy-time computation. At any given timestep, the agent observes its current state $s$ and queries the corresponding row in the populated Q-table. The optimal action is determined by simply selecting the action $a$ that yields the highest Q-value for that state, mathematically expressed as $\text{argmax}_a \hat{Q}(s,a)$. 

Because the expected future rewards have already been computed and stored during the learning phase, the agent produces ideal autonomous behavior using a highly efficient memory lookup rather than performing complex value calculations or environment simulations at runtime.